In [ ]:
# Import required libraries for data manipulation and preprocessing
import pandas as pd  # For working with dataframes and CSV files
import numpy as np  # For numerical operations and arrays
import pickle  # For serializing and storing Python objects

# Set pandas display options to show all columns when printing dataframes
pd.set_option("display.max_columns", None)

### Loading Master Provider Data

In [ ]:
# Read the Master Providers CSV file into a dataframe
df_provider = pd.read_csv("../Train_Test_Data/Train_Master_Providers.csv")
# Convert PotentialFraud column from categorical (No/Yes) to numeric (0/1)
df_provider['PotentialFraud'] = df_provider['PotentialFraud'].map({"No":0,"Yes":1})
# Print the shape of the dataframe (rows, columns)
print(df_provider.shape)
# Print the count of unique Provider IDs in the dataset
print("Unique Provider Ids: ", df_provider.Provider.nunique())
# Display the first 5 rows of the dataframe
df_provider.head()

(5410, 2)
Unique Provider Ids:  5410


,Provider,PotentialFraud
0,PRV51001,0
1,PRV51003,1
2,PRV51004,0
3,PRV51005,1
4,PRV51007,0


### Loading IP Data

In [ ]:
# Read the Inpatient data CSV file into a dataframe
df_ip = pd.read_csv("../Train_Test_Data/Train_Inpatientdata.csv")
# Print the shape of the dataframe (rows, columns)
print(df_ip.shape)
# Display the first 5 rows of the dataframe
df_ip.head()

(40474, 30)


,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,OtherPhysician,AdmissionDt,ClmAdmitDiagnosisCode,DeductibleAmtPaid,DischargeDt,DiagnosisGroupCode,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6
0,BENE11001,CLM46614,2009-04-12,2009-04-18,PRV55912,26000,PHY390922,NaN,NaN,2009-04-12,7866,1068.0,2009-04-18,201,1970,4019,5853,7843,2768,71590,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BENE11001,CLM66048,2009-08-31,2009-09-02,PRV55907,5000,PHY318495,PHY318495,NaN,2009-08-31,6186,1068.0,2009-09-02,750,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7092.0,NaN,NaN,NaN,NaN,NaN
2,BENE11001,CLM68358,2009-09-17,2009-09-20,PRV56046,5000,PHY372395,NaN,PHY324689,2009-09-17,29590,1068.0,2009-09-20,883,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BENE11011,CLM38412,2009-02-14,2009-02-22,PRV52405,5000,PHY369659,PHY392961,PHY349768,2009-02-14,431,1068.0,2009-02-22,067,43491,2762,7843,32723,V1041,4254,25062,40390,4019,NaN,331.0,NaN,NaN,NaN,NaN,NaN
4,BENE11014,CLM63689,2009-08-13,2009-08-30,PRV56614,10000,PHY379376,PHY398258,NaN,2009-08-13,78321,1068.0,2009-08-30,975,042,3051,34400,5856,42732,486,5119,29620,20300,NaN,3893.0,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Convert AdmissionDt column to datetime format, handling any errors by setting them to NaT
df_ip['AdmissionDt'] = pd.to_datetime(df_ip['AdmissionDt'], errors = 'coerce')
# Convert DischargeDt column to datetime format, handling any errors by setting them to NaT
df_ip['DischargeDt'] = pd.to_datetime(df_ip['DischargeDt'], errors = 'coerce')
# Calculate the number of days stayed in hospital by subtracting admission date from discharge date
df_ip['IP_Number_of_Days_in_Hospital'] = (df_ip['DischargeDt'] - df_ip['AdmissionDt'] ).dt.days

# Convert ClaimStartDt column to datetime format, handling any errors by setting them to NaT
df_ip['ClaimStartDt'] = pd.to_datetime(df_ip['ClaimStartDt'], errors = 'coerce')
# Convert ClaimEndDt column to datetime format, handling any errors by setting them to NaT
df_ip['ClaimEndDt'] = pd.to_datetime(df_ip['ClaimEndDt'], errors = 'coerce')
# Calculate the number of days taken to process the claim
df_ip['IP_Claim_Days'] = (df_ip['ClaimEndDt'] - df_ip['ClaimStartDt'] ).dt.days

# Drop unnecessary date columns and physician columns as they are no longer needed
df_ip.drop(columns = ['AdmissionDt','DischargeDt','ClaimStartDt','ClaimEndDt','AttendingPhysician','OperatingPhysician','OtherPhysician'], inplace = True)
# Print the shape of the dataframe after dropping columns
print(df_ip.shape)
# Display the first 5 rows of the modified dataframe
df_ip.head()

(40474, 25)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,ClmAdmitDiagnosisCode,DeductibleAmtPaid,DiagnosisGroupCode,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,IP_Number_of_Days_in_Hospital,IP_Claim_Days
0,BENE11001,CLM46614,PRV55912,26000,7866,1068.0,201,1970,4019,5853,7843,2768,71590,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,6
1,BENE11001,CLM66048,PRV55907,5000,6186,1068.0,750,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7092.0,NaN,NaN,NaN,NaN,NaN,2,2
2,BENE11001,CLM68358,PRV56046,5000,29590,1068.0,883,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3
3,BENE11011,CLM38412,PRV52405,5000,431,1068.0,067,43491,2762,7843,32723,V1041,4254,25062,40390,4019,NaN,331.0,NaN,NaN,NaN,NaN,NaN,8,8
4,BENE11014,CLM63689,PRV56614,10000,78321,1068.0,975,042,3051,34400,5856,42732,486,5119,29620,20300,NaN,3893.0,NaN,NaN,NaN,NaN,NaN,17,17


In [ ]:
# Count the number of non-null diagnosis codes for each claim (max 10 possible diagnosis codes)
df_ip['IP_Unique_Disease_Count'] = df_ip[['ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10']].count(axis =1)

# Count the number of non-null procedure codes for each claim (max 6 possible procedure codes)
df_ip['IP_Unique_Treatment_Count'] = df_ip[['ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6']].count(axis =1)

# Drop all individual diagnosis and procedure code columns as we have already extracted their counts
df_ip.drop(columns = ['ClmAdmitDiagnosisCode','DiagnosisGroupCode','ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10','ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6'], inplace = True)
# Print the shape of the dataframe after dropping columns
print(df_ip.shape)
# Display the first 5 rows of the modified dataframe
df_ip.head()

(40474, 9)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,DeductibleAmtPaid,IP_Number_of_Days_in_Hospital,IP_Claim_Days,IP_Unique_Disease_Count,IP_Unique_Treatment_Count
0,BENE11001,CLM46614,PRV55912,26000,1068.0,6,6,9,0
1,BENE11001,CLM66048,PRV55907,5000,1068.0,2,2,3,1
2,BENE11001,CLM68358,PRV56046,5000,1068.0,3,3,6,0
3,BENE11011,CLM38412,PRV52405,5000,1068.0,8,8,9,1
4,BENE11014,CLM63689,PRV56614,10000,1068.0,17,17,9,1


In [ ]:
# Rename IP-related columns to more descriptive names for clarity
df_ip.rename(columns = {'ClaimID' : 'IP_ClaimID',
                        'InscClaimAmtReimbursed' : 'IP_InscClaimAmtReimbursed',
                        'DeductibleAmtPaid' : 'IP_DeductibleAmtPaid'}, inplace = True)
# Print the shape of the dataframe
print(df_ip.shape)
# Display the first 5 rows of the renamed dataframe
df_ip.head()

(40474, 9)


,BeneID,IP_ClaimID,Provider,IP_InscClaimAmtReimbursed,IP_DeductibleAmtPaid,IP_Number_of_Days_in_Hospital,IP_Claim_Days,IP_Unique_Disease_Count,IP_Unique_Treatment_Count
0,BENE11001,CLM46614,PRV55912,26000,1068.0,6,6,9,0
1,BENE11001,CLM66048,PRV55907,5000,1068.0,2,2,3,1
2,BENE11001,CLM68358,PRV56046,5000,1068.0,3,3,6,0
3,BENE11011,CLM38412,PRV52405,5000,1068.0,8,8,9,1
4,BENE11014,CLM63689,PRV56614,10000,1068.0,17,17,9,1


### Loading OP Data

In [ ]:
# Read the Outpatient data CSV file into a dataframe
df_op = pd.read_csv("../Train_Test_Data/Train_Outpatientdata.csv")
# Drop unnecessary physician columns from the outpatient data
df_op.drop(columns= ['AttendingPhysician','OperatingPhysician','OtherPhysician'],inplace = True)
# Print the shape of the dataframe after dropping columns
print(df_op.shape)
# Display the first 5 rows of the outpatient dataframe
df_op.head()

(517737, 24)


,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,DeductibleAmtPaid,ClmAdmitDiagnosisCode
0,BENE11002,CLM624349,10/11/2009,10/11/2009,PRV56011,30,78943,V5866,V1272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,56409
1,BENE11003,CLM189947,2/12/2009,2/12/2009,PRV57610,80,6115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,79380
2,BENE11003,CLM438021,6/27/2009,6/27/2009,PRV57595,10,2723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
3,BENE11004,CLM121801,1/6/2009,1/6/2009,PRV56011,40,71988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
4,BENE11004,CLM150998,1/22/2009,1/22/2009,PRV56011,200,82382,30000,72887,4280,7197,V4577,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,71947


In [ ]:
# Convert ClaimStartDt column to datetime format for outpatient data
df_op['ClaimStartDt'] = pd.to_datetime(df_op['ClaimStartDt'], errors='coerce')
# Convert ClaimEndDt column to datetime format for outpatient data
df_op['ClaimEndDt'] = pd.to_datetime(df_op['ClaimEndDt'], errors='coerce')
# Calculate the number of days taken to process the outpatient claim
df_op['OP_Claim_Days'] = (df_op['ClaimEndDt'] - df_op['ClaimStartDt']).dt.days
# Drop the date columns after extracting the claim days duration
df_op.drop(columns = ['ClaimStartDt','ClaimEndDt'], inplace = True)
# Print the shape of the dataframe
print(df_op.shape)
# Display the last 5 rows of the modified dataframe
df_op.tail()

(517737, 23)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,DeductibleAmtPaid,ClmAdmitDiagnosisCode,OP_Claim_Days
517732,BENE159198,CLM510792,PRV53699,800,2163,V4575,53190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
517733,BENE159198,CLM551294,PRV53702,400,07041,5781,25000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
517734,BENE159198,CLM596444,PRV53676,60,V570,78079,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
517735,BENE159198,CLM636992,PRV53689,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
517736,BENE159198,CLM686139,PRV53689,80,78900,78609,4280,71946,3310,75311,2724,V103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,1


In [ ]:
# Count the number of non-null diagnosis codes for each outpatient claim (max 10 possible)
df_op['OP_Unique_Disease_Count'] = df_op[['ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10']].count(axis =1)

# Count the number of non-null procedure codes for each outpatient claim (max 6 possible)
df_op['OP_Unique_Treatment_Count'] = df_op[['ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6']].count(axis =1)

# Drop all individual diagnosis and procedure code columns to reduce dimensionality
df_op.drop(columns = ['ClmAdmitDiagnosisCode','ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10','ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6'], inplace = True)
# Print the shape of the dataframe after feature engineering
print(df_op.shape)
# Display the first 5 rows of the processed outpatient dataframe
df_op.head()

(517737, 8)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,DeductibleAmtPaid,OP_Claim_Days,OP_Unique_Disease_Count,OP_Unique_Treatment_Count
0,BENE11002,CLM624349,PRV56011,30,0,0,3,0
1,BENE11003,CLM189947,PRV57610,80,0,0,1,0
2,BENE11003,CLM438021,PRV57595,10,0,0,1,0
3,BENE11004,CLM121801,PRV56011,40,0,0,1,0
4,BENE11004,CLM150998,PRV56011,200,0,0,6,0


In [ ]:
# Rename outpatient-related columns with OP prefix for clarity when merging datasets
df_op.rename(columns = {'ClaimID' : 'OP_ClaimID',
                        'InscClaimAmtReimbursed' : 'OP_InscClaimAmtReimbursed',
                        'DeductibleAmtPaid' : 'OP_DeductibleAmtPaid'}, inplace = True)
# Print the shape of the renamed dataframe
print(df_op.shape)
# Display the first 5 rows of the outpatient dataframe with renamed columns
df_op.head()

(517737, 8)


,BeneID,OP_ClaimID,Provider,OP_InscClaimAmtReimbursed,OP_DeductibleAmtPaid,OP_Claim_Days,OP_Unique_Disease_Count,OP_Unique_Treatment_Count
0,BENE11002,CLM624349,PRV56011,30,0,0,3,0
1,BENE11003,CLM189947,PRV57610,80,0,0,1,0
2,BENE11003,CLM438021,PRV57595,10,0,0,1,0
3,BENE11004,CLM121801,PRV56011,40,0,0,1,0
4,BENE11004,CLM150998,PRV56011,200,0,0,6,0


### Merging IP Date with Master Provider Data

In [ ]:
# Merge inpatient data with provider data on Provider column using a left join to keep all providers
df_ip_provider = df_provider[['Provider']].merge(df_ip, on = 'Provider', how = 'left')
# Print the shape of the merged dataframe
print(df_ip_provider.shape)
# Display the first 2 rows of the merged dataframe
df_ip_provider.head(2)

(43792, 9)


,Provider,BeneID,IP_ClaimID,IP_InscClaimAmtReimbursed,IP_DeductibleAmtPaid,IP_Number_of_Days_in_Hospital,IP_Claim_Days,IP_Unique_Disease_Count,IP_Unique_Treatment_Count
0,PRV51001,BENE36012,CLM58316,36000.0,1068.0,4.0,4.0,8.0,0.0
1,PRV51001,BENE38773,CLM52334,12000.0,1068.0,2.0,2.0,6.0,0.0


In [ ]:
# Aggregate inpatient data at provider level using different aggregation functions
df_ip_final = df_ip_provider.groupby('Provider', as_index=False).agg({
    "IP_ClaimID" : lambda x : x.nunique(),  # Count unique inpatient claims per provider
    "BeneID" : lambda x : x.nunique(),  # Count unique beneficiaries per provider
    'IP_InscClaimAmtReimbursed' : lambda x: x.mean(),  # Average insurance reimbursement amount
    'IP_DeductibleAmtPaid' : lambda x : x.mean(),  # Average deductible paid per claim
    'IP_Number_of_Days_in_Hospital': lambda x: x.mode().iloc[0] if not x.mode().empty else None,  # Most common hospital stay duration
    'IP_Claim_Days': lambda x: x.mode().iloc[0] if not x.mode().empty else None,  # Most common claim processing days
    'IP_Unique_Disease_Count': lambda x: x.mode().iloc[0] if not x.mode().empty else None,  # Most common disease count
    'IP_Unique_Treatment_Count': lambda x: x.mode().iloc[0] if not x.mode().empty else None  # Most common treatment count
})

# Rename aggregated columns to have more descriptive names
df_ip_final.rename(columns = {
    "IP_ClaimID":"IP_Claim_Count", 
    "BeneID" : "IP_Benf_Count",
    'IP_Number_of_Days_in_Hospital': "Avg_IP_Number_of_Days_in_Hospital",
    'IP_Claim_Days': "Avg_IP_Number_of_Days_in_Hospital",
    'IP_Unique_Disease_Count': "Avg_IP_Number_of_Days_in_Hospital",
    'IP_Unique_Treatment_Count': "Avg_IP_Number_of_Days_in_Hospital" ,
    'IP_InscClaimAmtReimbursed' : "Avg_IP_InscClaimAmtReimbursed",
    "IP_DeductibleAmtPaid":"Avg_IP_DeductibleAmtPaid"
}, inplace= True)
# Print the shape of the final IP dataframe
print(df_ip_final.shape)
# Display the first 5 rows of aggregated IP data
df_ip_final.head()

(5410, 9)


,Provider,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital
0,PRV51001,5,5,19400.000000,1068.0,0.0,0.0,6.0,0.0
1,PRV51003,62,53,9241.935484,1068.0,3.0,3.0,9.0,1.0
2,PRV51004,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,PRV51005,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,PRV51007,3,3,6333.333333,1068.0,4.0,4.0,6.0,0.0


### Merging OP Data with Master Provider Data

In [ ]:
# Merge outpatient data with provider data on Provider column using a left join
df_op_provider=df_provider[['Provider']].merge(df_op,on='Provider',how='left')
# Print the shape of the merged dataframe
print(df_op_provider.shape)
# Display the first 5 rows of the merged dataframe
df_op_provider.head()

(518135, 8)


,Provider,BeneID,OP_ClaimID,OP_InscClaimAmtReimbursed,OP_DeductibleAmtPaid,OP_Claim_Days,OP_Unique_Disease_Count,OP_Unique_Treatment_Count
0,PRV51001,BENE11727,CLM733300,20.0,0.0,0.0,2.0,0.0
1,PRV51001,BENE24646,CLM372475,700.0,0.0,1.0,6.0,0.0
2,PRV51001,BENE31617,CLM748221,900.0,0.0,0.0,1.0,0.0
3,PRV51001,BENE32715,CLM272936,500.0,0.0,1.0,6.0,0.0
4,PRV51001,BENE49220,CLM452024,70.0,0.0,0.0,1.0,0.0


### 

In [ ]:
# Aggregate outpatient data at provider level using different aggregation functions
df_op_final = df_op_provider.groupby('Provider', as_index=False).agg({
    "OP_ClaimID" : lambda x : x.nunique(),  # Count unique outpatient claims per provider
    "BeneID" : lambda x : x.nunique(),  # Count unique beneficiaries per provider
    'OP_InscClaimAmtReimbursed' : lambda x: x.mean(),  # Average insurance reimbursement amount
    'OP_DeductibleAmtPaid' : lambda x : x.mean(),  # Average deductible paid per claim
    'OP_Claim_Days': lambda x: x.mode().iloc[0] if not x.mode().empty else None,  # Most common claim processing days
    'OP_Unique_Disease_Count': lambda x: x.mode().iloc[0] if not x.mode().empty else None,  # Most common disease count
    'OP_Unique_Treatment_Count': lambda x: x.mode().iloc[0] if not x.mode().empty else None  # Most common treatment count
})

# Rename aggregated columns with more descriptive names
df_op_final.rename(columns = {
    "OP_ClaimID":"OP_Claim_Count", 
    "BeneID" : "OP_Benf_Count",
    'OP_Claim_Days': "Avg_OP_Number_of_Days_in_Hospital",
    'OP_Unique_Disease_Count': "Avg_OP_Number_of_Days_in_Hospital",
    'OP_Unique_Treatment_Count': "Avg_OP_Number_of_Days_in_Hospital" ,
    'OP_InscClaimAmtReimbursed' : "Avg_OP_InscClaimAmtReimbursed",
    "OP_DeductibleAmtPaid":"Avg_OP_DeductibleAmtPaid"
}, inplace= True)
# Print the shape of the final OP dataframe
print(df_op_final.shape)
# Display the first 5 rows of aggregated OP data
df_op_final.head()

(5410, 8)


,Provider,OP_Claim_Count,OP_Benf_Count,Avg_OP_InscClaimAmtReimbursed,Avg_OP_DeductibleAmtPaid,Avg_OP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital
0,PRV51001,20,19,382.000000,0.000000,0.0,1.0,0.0
1,PRV51003,70,66,466.714286,1.000000,0.0,1.0,0.0
2,PRV51004,149,138,350.134228,2.080537,0.0,1.0,0.0
3,PRV51005,1165,495,241.124464,3.175966,0.0,1.0,0.0
4,PRV51007,69,56,213.188406,0.869565,0.0,1.0,0.0


In [ ]:
# Check for missing values (NaN) in the outpatient final dataframe
df_op_final.isna().sum()

Provider                               0
OP_Claim_Count                         0
OP_Benf_Count                          0
Avg_OP_InscClaimAmtReimbursed        398
Avg_OP_DeductibleAmtPaid             398
Avg_OP_Number_of_Days_in_Hospital    398
Avg_OP_Number_of_Days_in_Hospital    398
Avg_OP_Number_of_Days_in_Hospital    398
dtype: int64

### Loading Benefitiory Data

In [ ]:
# Read the beneficiary data CSV file into a dataframe
df_benf = pd.read_csv("../Train_Test_Data/Train_Beneficiarydata.csv")
# Drop unnecessary demographic columns that are not useful for fraud detection modeling
df_benf.drop(columns = ['DOB','DOD','Gender','Race','State','County'], inplace = True)
# Print the shape of the beneficiary dataframe
print(df_benf.shape)
# Display the first 5 rows of the beneficiary dataframe
df_benf.head()

(138556, 19)


,BeneID,RenalDiseaseIndicator,NoOfMonths_PartACov,NoOfMonths_PartBCov,ChronicCond_Alzheimer,ChronicCond_Heartfailure,ChronicCond_KidneyDisease,ChronicCond_Cancer,ChronicCond_ObstrPulmonary,ChronicCond_Depression,ChronicCond_Diabetes,ChronicCond_IschemicHeart,ChronicCond_Osteoporasis,ChronicCond_rheumatoidarthritis,ChronicCond_stroke,IPAnnualReimbursementAmt,IPAnnualDeductibleAmt,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt
0,BENE11001,0,12,12,1,2,1,2,2,1,1,1,2,1,1,36000,3204,60,70
1,BENE11002,0,12,12,2,2,2,2,2,2,2,2,2,2,2,0,0,30,50
2,BENE11003,0,12,12,1,2,2,2,2,2,2,1,2,2,2,0,0,90,40
3,BENE11004,0,12,12,1,1,2,2,2,2,1,1,1,1,2,0,0,1810,760
4,BENE11005,0,12,12,2,2,2,2,1,2,1,2,2,2,2,0,0,1790,1200


In [ ]:
# Define list of chronic condition columns that need to be converted from categorical to numeric
mapping_cols = ['ChronicCond_Alzheimer', 'ChronicCond_Heartfailure',
       'ChronicCond_KidneyDisease', 'ChronicCond_Cancer',
       'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression',
       'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart',
       'ChronicCond_Osteoporasis', 'ChronicCond_rheumatoidarthritis',
       'ChronicCond_stroke']

# Convert chronic condition columns from (1=No, 2=Yes) to (0=No, 1=Yes) format
for c in mapping_cols:
    df_benf[c] = df_benf[c].map({1:0,2:1})

# Convert RenalDiseaseIndicator from string ('0'=No, 'Y'=Yes) to numeric (0/1) format
df_benf['RenalDiseaseIndicator'] = df_benf['RenalDiseaseIndicator'].map({'0':0,'Y':1})
# Print the shape of the processed beneficiary dataframe
print(df_benf.shape)
# Display the first 5 rows with converted categorical values
df_benf.head()

(138556, 19)


,BeneID,RenalDiseaseIndicator,NoOfMonths_PartACov,NoOfMonths_PartBCov,ChronicCond_Alzheimer,ChronicCond_Heartfailure,ChronicCond_KidneyDisease,ChronicCond_Cancer,ChronicCond_ObstrPulmonary,ChronicCond_Depression,ChronicCond_Diabetes,ChronicCond_IschemicHeart,ChronicCond_Osteoporasis,ChronicCond_rheumatoidarthritis,ChronicCond_stroke,IPAnnualReimbursementAmt,IPAnnualDeductibleAmt,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt
0,BENE11001,0,12,12,0,1,0,1,1,0,0,0,1,0,0,36000,3204,60,70
1,BENE11002,0,12,12,1,1,1,1,1,1,1,1,1,1,1,0,0,30,50
2,BENE11003,0,12,12,0,1,1,1,1,1,1,0,1,1,1,0,0,90,40
3,BENE11004,0,12,12,0,0,1,1,1,1,0,0,0,0,1,0,0,1810,760
4,BENE11005,0,12,12,1,1,1,1,0,1,0,1,1,1,1,0,0,1790,1200


In [ ]:
# Combine Provider and BeneID columns from both inpatient and outpatient dataframes into a single dataframe
df_ip_op_benf = pd.concat([df_ip[['Provider','BeneID']], df_op[['Provider','BeneID']]], axis = 0)
# Print the shape before removing duplicates
print("before removing duplicates :",df_ip_op_benf.shape)
# Remove duplicate Provider-BeneID combinations and reset the index
df_ip_op_benf_unique = df_ip_op_benf.drop_duplicates().reset_index(drop = True).copy()
# Print the shape after removing duplicates
print("after removing duplicates :",df_ip_op_benf_unique.shape)
# Display the first 5 rows of unique Provider-BeneID pairs
df_ip_op_benf_unique.head()

before removing duplicates : (558211, 2)
after removing duplicates : (363300, 2)


,Provider,BeneID
0,PRV55912,BENE11001
1,PRV55907,BENE11001
2,PRV56046,BENE11001
3,PRV52405,BENE11011
4,PRV56614,BENE11014


### Merging Benf Data with Provider Id

In [ ]:
# Merge beneficiary data with provider and beneficiary IDs on BeneID column using left join
df_benf_provider  = df_ip_op_benf_unique.merge(df_benf, on = 'BeneID', how = 'left')
# Print the shape of the merged dataframe
print(df_benf_provider.shape)
# Display the first 5 rows of the merged beneficiary and provider data
df_benf_provider.head()

(363300, 20)


,Provider,BeneID,RenalDiseaseIndicator,NoOfMonths_PartACov,NoOfMonths_PartBCov,ChronicCond_Alzheimer,ChronicCond_Heartfailure,ChronicCond_KidneyDisease,ChronicCond_Cancer,ChronicCond_ObstrPulmonary,ChronicCond_Depression,ChronicCond_Diabetes,ChronicCond_IschemicHeart,ChronicCond_Osteoporasis,ChronicCond_rheumatoidarthritis,ChronicCond_stroke,IPAnnualReimbursementAmt,IPAnnualDeductibleAmt,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt
0,PRV55912,BENE11001,0,12,12,0,1,0,1,1,0,0,0,1,0,0,36000,3204,60,70
1,PRV55907,BENE11001,0,12,12,0,1,0,1,1,0,0,0,1,0,0,36000,3204,60,70
2,PRV56046,BENE11001,0,12,12,0,1,0,1,1,0,0,0,1,0,0,36000,3204,60,70
3,PRV52405,BENE11011,0,12,12,1,0,0,1,1,0,0,1,1,0,0,5000,1068,250,320
4,PRV56614,BENE11014,1,12,12,1,0,0,1,0,0,1,0,1,1,1,21260,2136,120,100


In [ ]:
# Aggregate beneficiary data at provider level
df_benf_final = df_benf_provider.groupby("Provider", as_index=False).agg(
    Total_Beneficiaries=("BeneID", "count"),  # Total number of beneficiaries per provider
    RenalDisease_Count=("RenalDiseaseIndicator", "sum"),  # Count of beneficiaries with renal disease
    Alzheimer_Count=("ChronicCond_Alzheimer", "sum"),  # Count of beneficiaries with Alzheimer's
    HeartFailure_Count=("ChronicCond_Heartfailure", "sum"),  # Count of beneficiaries with heart failure
    KidneyDisease_Count=("ChronicCond_KidneyDisease", "sum"),  # Count of beneficiaries with kidney disease
    Cancer_Count=("ChronicCond_Cancer", "sum"),  # Count of beneficiaries with cancer
    Pulmonary_Count=("ChronicCond_ObstrPulmonary", "sum"),  # Count of beneficiaries with pulmonary disease
    Depression_Count=("ChronicCond_Depression", "sum"),  # Count of beneficiaries with depression
    Diabetes_Count=("ChronicCond_Diabetes", "sum"),  # Count of beneficiaries with diabetes
    IschemicHeart_Count=("ChronicCond_IschemicHeart", "sum"),  # Count of beneficiaries with ischemic heart disease
    Osteoporosis_Count=("ChronicCond_Osteoporasis", "sum"),  # Count of beneficiaries with osteoporosis
    RheumatoidArthritis_Count=("ChronicCond_rheumatoidarthritis", "sum"),  # Count of beneficiaries with rheumatoid arthritis
    Stroke_Count=("ChronicCond_stroke", "sum"),  # Count of beneficiaries with stroke
    Avg_PartA_Months=("NoOfMonths_PartACov", lambda x: x.mode().iloc[0] if not x.mode().empty else None),  # Most common Part A coverage months
    Avg_PartB_Months_Mode=("NoOfMonths_PartBCov", lambda x: x.mode().iloc[0] if not x.mode().empty else None),  # Most common Part B coverage months
    Avg_IP_Reimbursement=("IPAnnualReimbursementAmt", "mean"),  # Average annual inpatient reimbursement
    Avg_IP_Deductible=("IPAnnualDeductibleAmt", "mean"),  # Average annual inpatient deductible
    Avg_OP_Reimbursement=("OPAnnualReimbursementAmt", "mean"),  # Average annual outpatient reimbursement
    Avg_OP_Deductible=("OPAnnualDeductibleAmt", "mean"),  # Average annual outpatient deductible
)
# Print the shape of the final beneficiary dataframe
print(df_benf_final.shape)
# Display the first 5 rows of aggregated beneficiary data
df_benf_final.head()

(5410, 20)


,Provider,Total_Beneficiaries,RenalDisease_Count,Alzheimer_Count,HeartFailure_Count,KidneyDisease_Count,Cancer_Count,Pulmonary_Count,Depression_Count,Diabetes_Count,IschemicHeart_Count,Osteoporosis_Count,RheumatoidArthritis_Count,Stroke_Count,Avg_PartA_Months,Avg_PartB_Months_Mode,Avg_IP_Reimbursement,Avg_IP_Deductible,Avg_OP_Reimbursement,Avg_OP_Deductible
0,PRV51001,24,8,10,6,7,19,15,15,4,2,18,16,19,12,12,18047.916667,890.000000,2537.500000,474.916667
1,PRV51003,117,22,73,47,65,107,84,70,30,18,89,85,108,12,12,6814.017094,822.632479,2490.598291,664.529915
2,PRV51004,138,20,78,56,91,122,101,78,42,40,95,97,122,12,12,4596.739130,454.144928,2095.144928,600.869565
3,PRV51005,495,79,330,232,317,436,390,311,181,148,353,368,456,12,12,3717.232323,398.698990,1798.808081,475.965657
4,PRV51007,58,9,37,28,41,52,46,37,22,18,41,42,49,12,12,3109.655172,423.517241,1497.241379,430.689655


### Merging all Data (IP + OU + Benf + Target)

In [ ]:
# Merge provider data with aggregated IP data on Provider column
df_final = df_provider.merge(df_ip_final,on = 'Provider', how = 'left')
# Merge the result with aggregated OP data on Provider column
df_final = df_final.merge(df_op_final,on = 'Provider', how = 'left')
# Merge the result with aggregated beneficiary data on Provider column
df_final = df_final.merge(df_benf_final,on = 'Provider', how = 'left')
# Print the shape of the final combined dataframe
print(df_final.shape)
# Display the first 5 rows of the final merged dataset
df_final.head()

(5410, 36)


,Provider,PotentialFraud,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,OP_Claim_Count,OP_Benf_Count,Avg_OP_InscClaimAmtReimbursed,Avg_OP_DeductibleAmtPaid,Avg_OP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital,Total_Beneficiaries,RenalDisease_Count,Alzheimer_Count,HeartFailure_Count,KidneyDisease_Count,Cancer_Count,Pulmonary_Count,Depression_Count,Diabetes_Count,IschemicHeart_Count,Osteoporosis_Count,RheumatoidArthritis_Count,Stroke_Count,Avg_PartA_Months,Avg_PartB_Months_Mode,Avg_IP_Reimbursement,Avg_IP_Deductible,Avg_OP_Reimbursement,Avg_OP_Deductible
0,PRV51001,0,5,5,19400.000000,1068.0,0.0,0.0,6.0,0.0,20,19,382.000000,0.000000,0.0,1.0,0.0,24,8,10,6,7,19,15,15,4,2,18,16,19,12,12,18047.916667,890.000000,2537.500000,474.916667
1,PRV51003,1,62,53,9241.935484,1068.0,3.0,3.0,9.0,1.0,70,66,466.714286,1.000000,0.0,1.0,0.0,117,22,73,47,65,107,84,70,30,18,89,85,108,12,12,6814.017094,822.632479,2490.598291,664.529915
2,PRV51004,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,149,138,350.134228,2.080537,0.0,1.0,0.0,138,20,78,56,91,122,101,78,42,40,95,97,122,12,12,4596.739130,454.144928,2095.144928,600.869565
3,PRV51005,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,1165,495,241.124464,3.175966,0.0,1.0,0.0,495,79,330,232,317,436,390,311,181,148,353,368,456,12,12,3717.232323,398.698990,1798.808081,475.965657
4,PRV51007,0,3,3,6333.333333,1068.0,4.0,4.0,6.0,0.0,69,56,213.188406,0.869565,0.0,1.0,0.0,58,9,37,28,41,52,46,37,22,18,41,42,49,12,12,3109.655172,423.517241,1497.241379,430.689655


In [ ]:
# Replace all missing values (NaN) in the dataframe with 0
df_final.fillna(0,inplace=True)

In [ ]:
# Check if there are any remaining missing values in the entire dataframe
df_final.isna().sum().sum()

0

In [ ]:
# Save the final processed dataframe to a CSV file for use in model building
df_final.to_csv("../Model_Input_Data/Model_Input_Data.csv",index=False)